# Analyzing a Book Database with SQL

**Author:** Luciana Jost

**TripleTen Data Analyst Certification**

**Date:** October 2026

> **About this version:** this is the English translation of the original analysis, written in Portuguese (available in the `pt/` folder). The database belongs to the bootcamp environment and cannot be queried again, so the code and the results shown are exactly those of the original run. For that reason, the column names created in the SQL queries are kept in Portuguese. Glossary: `livros_lancados_depois_de_2000` = books released after 2000; `numero_de_avaliacoes` = number of reviews; `classificacao_media` = average rating; `editora` = publisher; `numero_de_livros` = number of books; `autor` = author; `livros_considerados` = books counted; `usuarios_com_mais_de_50_classificacoes` = users with more than 50 ratings; `total_de_livros` = total books; `livros_sem_avaliacao` = books without reviews; `maximo_de_avaliacoes` = highest number of reviews; `media_de_avaliacoes_por_livro` = average reviews per book; `media_das_classificacoes` = average rating; `media_de_avaliacoes` = average number of reviews. In the printed output of section 2.2, "Tabela" means table and "linhas" means rows.

## 1. Study goals

With the change in routine brought by the pandemic, reading took up more of people's free time, and startups began building apps for readers. This study analyzes the database of a competing service in this market, with information about books, authors, publishers, ratings and user reviews.

The goal is to extract insights that help shape the value proposition of a new product. To do so, the study answers five questions:

1. How many books were released after January 1, 2000?
2. What is the number of reviews and the average rating of each book?
3. Which publisher released the most books with more than 50 pages?
4. Which author has the highest average rating, considering only books with at least 50 ratings?
5. What is the average number of reviews among users who rated more than 50 books?

**About the terms:** in this study, a "rating" is the score a user gives a book, stored in the `ratings` table. A "review" is the text a user writes about a book, stored in the `reviews` table.

### Structure of the analysis

1. Study goals
2. Connecting to the database and exploring the tables
3. Queries
4. Conclusions

## 2. Connecting to the database and exploring the tables

### 2.1 Connection

The database connection follows the instructions provided by the bootcamp. The `executar_consulta` function ("run query") takes a SQL query and returns the result as a DataFrame. pandas is only used to run the queries and display the results.

**About credentials:** for security, the database user, password and address are not written in the notebook. They are read from environment variables (`DB_USER`, `DB_PASSWORD`, `DB_HOST`, `DB_PORT` and `DB_NAME`), defined outside the project. The database belongs to the bootcamp environment and is not public, so the results shown in this notebook are those of the original run.

In [29]:
import os

import pandas as pd
from sqlalchemy import create_engine

# Credentials are read from environment variables and are never stored in the code
db_config = {
    'user': os.environ['DB_USER'],
    'pwd': os.environ['DB_PASSWORD'],
    'host': os.environ['DB_HOST'],
    'port': int(os.environ.get('DB_PORT', 5432)),
    'db': os.environ['DB_NAME'],
}

connection_string = 'postgresql://{}:{}@{}:{}/{}'.format(
    db_config['user'],
    db_config['pwd'],
    db_config['host'],
    db_config['port'],
    db_config['db'],
)

engine = create_engine(connection_string, connect_args={'sslmode': 'require'})


def executar_consulta(query):
    """Runs a SQL query on the database.

    Takes the query text and returns the result as a DataFrame.
    """
    return pd.io.sql.read_sql(query, con=engine)

### 2.2 Exploring the tables

The database has five tables. The cell below shows the number of rows and the first rows of each one.

In [30]:
TABELAS = ['books', 'authors', 'publishers', 'ratings', 'reviews']

for tabela in TABELAS:
    total = executar_consulta(f'SELECT COUNT(*) AS total FROM {tabela}')
    print(f'Tabela: {tabela} ({total.loc[0, "total"]} linhas)')
    display(executar_consulta(f'SELECT * FROM {tabela} LIMIT 5'))

Tabela: books (1000 linhas)


,book_id,author_id,title,num_pages,publication_date,publisher_id
0,1,546,'Salem's Lot,594,2005-11-01,93
1,2,465,1 000 Places to See Before You Die,992,2003-05-22,336
2,3,407,13 Little Blue Envelopes (Little Blue Envelope...,322,2010-12-21,135
3,4,82,1491: New Revelations of the Americas Before C...,541,2006-10-10,309
4,5,125,1776,386,2006-07-04,268


Tabela: authors (636 linhas)


,author_id,author
0,1,A.S. Byatt
1,2,Aesop/Laura Harris/Laura Gibbs
2,3,Agatha Christie
3,4,Alan Brennert
4,5,Alan Moore/David Lloyd


Tabela: publishers (340 linhas)


,publisher_id,publisher
0,1,Ace
1,2,Ace Book
2,3,Ace Books
3,4,Ace Hardcover
4,5,Addison Wesley Publishing Company


Tabela: ratings (6456 linhas)


,rating_id,book_id,username,rating
0,1,1,ryanfranco,4
1,2,1,grantpatricia,2
2,3,1,brandtandrea,5
3,4,2,lorichen,3
4,5,2,mariokeller,2


Tabela: reviews (2793 linhas)


,review_id,book_id,username,text
0,1,1,brandtandrea,Mention society tell send professor analysis. ...
1,2,1,ryanfranco,Foot glass pretty audience hit themselves. Amo...
2,3,2,lorichen,Listen treat keep worry. Miss husband tax but ...
3,4,3,johnsonamanda,Finally month interesting blue could nature cu...
4,5,3,scotttamara,Nation purpose heavy give wait song will. List...


**Reading the tables:**

- `books` has 1,000 books, with title, number of pages, publication date and the author and publisher identifiers.
- `authors` has 636 authors. In some records, the field holds more than one name, separated by a slash, as in works with co-authors or illustrators.
- `publishers` has 340 publishers. Some names are very similar, such as "Ace", "Ace Book" and "Ace Books", recorded as different publishers.
- `ratings` has 6,456 ratings, each with the book, the user and the score given.
- `reviews` has 2,793 reviews, each with the book, the user and the written text.

The tables are linked by identifiers: `book_id` links `books` to `ratings` and `reviews`, `author_id` links `books` to `authors` and `publisher_id` links `books` to `publishers`.

The number of ratings (6,456) is higher than the number of reviews (2,793): not every user who gives a score writes a text.

## 3. Queries

### 3.1 Books released after January 1, 2000

The query counts the books whose publication date is later than January 1, 2000.

In [31]:
query = '''
SELECT
    COUNT(*) AS livros_lancados_depois_de_2000
FROM books
WHERE publication_date > '2000-01-01';
'''

executar_consulta(query)

,livros_lancados_depois_de_2000
0,819


**Result:** 819 books were released after January 1, 2000.

**Conclusion:** of the 1,000 books in the catalog, 819 (81.9%) were published after January 1, 2000, and books published up to that date make up less than a fifth of the total. The competitor's catalog consists mainly of recent editions. This suggests that titles from the last two decades should carry significant weight in a competitive catalog. The data, however, show what the competitor offers, not what readers are looking for: confirming this priority would require usage or sales data.

### 3.2 Number of reviews and average rating of each book

For each book, the query calculates the number of written reviews (`reviews` table) and the average score received (`ratings` table). The two calculations are done in separate subqueries and then joined to the books table, so that every book appears in the result, including those without reviews.

In [32]:
query = '''
SELECT
    b.book_id,
    b.title,
    COALESCE(rv.numero_de_avaliacoes, 0) AS numero_de_avaliacoes,
    ROUND(rt.classificacao_media, 2) AS classificacao_media
FROM books AS b
LEFT JOIN (
    SELECT
        book_id,
        COUNT(review_id) AS numero_de_avaliacoes
    FROM reviews
    GROUP BY book_id
) AS rv ON rv.book_id = b.book_id
LEFT JOIN (
    SELECT
        book_id,
        AVG(rating) AS classificacao_media
    FROM ratings
    GROUP BY book_id
) AS rt ON rt.book_id = b.book_id
ORDER BY numero_de_avaliacoes DESC, classificacao_media DESC;
'''

avaliacoes_por_livro = executar_consulta(query)
avaliacoes_por_livro

,book_id,title,numero_de_avaliacoes,classificacao_media
0,948,Twilight (Twilight #1),7,3.66
1,302,Harry Potter and the Prisoner of Azkaban (Harr...,6,4.41
2,299,Harry Potter and the Chamber of Secrets (Harry...,6,4.29
3,656,The Book Thief,6,4.26
4,734,The Glass Castle,6,4.21
...,...,...,...,...
995,387,Leonardo's Notebooks,0,4.00
996,191,Disney's Beauty and the Beast (A Little Golden...,0,4.00
997,221,Essential Tales and Poems,0,4.00
998,83,Anne Rice's The Vampire Lestat: A Graphic Novel,0,3.67


Since the result has one row per book, the next query summarizes the table: total books, books without reviews, highest number of reviews, average reviews per book and average rating.

In [33]:
query = '''
WITH por_livro AS (
    SELECT
        b.book_id,
        COALESCE(rv.numero_de_avaliacoes, 0) AS numero_de_avaliacoes,
        rt.classificacao_media
    FROM books AS b
    LEFT JOIN (
        SELECT
            book_id,
            COUNT(review_id) AS numero_de_avaliacoes
        FROM reviews
        GROUP BY book_id
    ) AS rv ON rv.book_id = b.book_id
    LEFT JOIN (
        SELECT
            book_id,
            AVG(rating) AS classificacao_media
        FROM ratings
        GROUP BY book_id
    ) AS rt ON rt.book_id = b.book_id
)
SELECT
    COUNT(*) AS total_de_livros,
    SUM(CASE WHEN numero_de_avaliacoes = 0 THEN 1 ELSE 0 END) AS livros_sem_avaliacao,
    MAX(numero_de_avaliacoes) AS maximo_de_avaliacoes,
    ROUND(AVG(numero_de_avaliacoes), 2) AS media_de_avaliacoes_por_livro,
    ROUND(AVG(classificacao_media), 2) AS media_das_classificacoes
FROM por_livro;
'''

executar_consulta(query)

,total_de_livros,livros_sem_avaliacao,maximo_de_avaliacoes,media_de_avaliacoes_por_livro,media_das_classificacoes
0,1000,6,7,2.79,3.9


**Result:** the query returns one row for each of the 1,000 books, with the number of reviews and the average rating.

- The number of reviews per book ranges from 0 to 7, with an average of 2.79.
- The most reviewed book is *Twilight (Twilight #1)*, with 7 reviews and an average rating of 3.66.
- Six books have no written reviews.
- The average of the books' average ratings is 3.90, on a scale up to 5. The `media_das_classificacoes` column in the summary is therefore an average of the per-book averages, where each book has the same weight.

**Conclusion:** users write few reviews: on average, fewer than three per book. The most reviewed titles are popular, mass-market works, such as *Twilight* and the *Harry Potter* series. The book with the most reviews, however, is not the best rated: *Twilight* averages 3.66, below the overall average. Review volume and perceived quality are different measures, and the new product should track both. Ratings are generally high, which may suggest that users tend to rate books they liked, a hypothesis these data cannot confirm.

### 3.3 Publisher with the most books over 50 pages

The query only considers books with more than 50 pages, to exclude pamphlets and similar publications, and counts how many books each publisher released. The result shows the top five publishers, to give context to the leader.

In [34]:
query = '''
SELECT
    p.publisher AS editora,
    COUNT(b.book_id) AS numero_de_livros
FROM books AS b
INNER JOIN publishers AS p ON p.publisher_id = b.publisher_id
WHERE b.num_pages > 50
GROUP BY p.publisher_id, p.publisher
ORDER BY numero_de_livros DESC
LIMIT 5;
'''

executar_consulta(query)

,editora,numero_de_livros
0,Penguin Books,42
1,Vintage,31
2,Grand Central Publishing,25
3,Penguin Classics,24
4,Bantam,19


**Result:** the publisher that released the most books with more than 50 pages is **Penguin Books**, with 42 books.

It is followed by Vintage (31 books), Grand Central Publishing (25), Penguin Classics (24) and Bantam (19).

**Conclusion:** Penguin Books leads by a wide margin: it has 11 more books than the runner-up. Fourth place goes to Penguin Classics, recorded in the database as a separate publisher. Together, the two Penguin imprints account for 66 books, which reinforces the weight of this publishing group in the catalog. As noted when exploring the tables, the database records similar names as different publishers, so the real concentration by publishing group may be higher than the count by name shows. For the new product, the publishers at the top of the list are the most relevant catalog partnerships.

### 3.4 Author with the highest average rating

The query only considers books with at least 50 ratings. For each of them, it calculates the average rating. It then averages these ratings by author. The result shows the five best-rated authors and how many of each author's books were counted.

In [35]:
query = '''
SELECT
    a.author AS autor,
    COUNT(lv.book_id) AS livros_considerados,
    ROUND(AVG(lv.classificacao_media), 2) AS classificacao_media
FROM (
    SELECT
        book_id,
        AVG(rating) AS classificacao_media
    FROM ratings
    GROUP BY book_id
    HAVING COUNT(rating_id) >= 50
) AS lv
INNER JOIN books AS b ON b.book_id = lv.book_id
INNER JOIN authors AS a ON a.author_id = b.author_id
GROUP BY a.author_id, a.author
ORDER BY AVG(lv.classificacao_media) DESC
LIMIT 5;
'''

executar_consulta(query)

,autor,livros_considerados,classificacao_media
0,J.K. Rowling/Mary GrandPré,4,4.28
1,Markus Zusak/Cao Xuân Việt Khương,1,4.26
2,J.R.R. Tolkien,2,4.26
3,Louisa May Alcott,1,4.19
4,Rick Riordan,1,4.08


**Result:** the author with the highest average rating, among books with at least 50 ratings, is **J.K. Rowling/Mary GrandPré**, with an average of 4.28 across four books.

They are followed by Markus Zusak/Cao Xuân Việt Khương (4.26, one book), J.R.R. Tolkien (4.26, two books), Louisa May Alcott (4.19, one book) and Rick Riordan (4.08, one book).

Each author's rating was calculated as the average of their books' average ratings, so each book has the same weight in the result.

**Conclusion:** J.K. Rowling, recorded together with illustrator Mary GrandPré, leads with the highest average and the largest number of books counted. Her result is the most solid on the list: the average across her four books is 4.28, while three of the five authors are counted with a single book. The differences at the top are small, two hundredths between first and second place.

Note that the filter is by book, not by user: the list shows the best-rated authors among books with at least 50 ratings. Since the database has no genre column, any reading by genre comes from outside knowledge and should be treated with caution. Part of the list is fantasy and young adult fiction (J.K. Rowling, J.R.R. Tolkien and Rick Riordan), but not all of it: Louisa May Alcott and Markus Zusak wrote novels that are not fantasy.

### 3.5 Average number of reviews among users who rated more than 50 books

The query identifies users who rated more than 50 books, counts how many written reviews each of them wrote and calculates the average among them. The result also shows how many users meet the criterion.

In [36]:
query = '''
SELECT
    COUNT(*) AS usuarios_com_mais_de_50_classificacoes,
    ROUND(AVG(numero_de_avaliacoes), 2) AS media_de_avaliacoes
FROM (
    SELECT
        u.username,
        COUNT(rv.review_id) AS numero_de_avaliacoes
    FROM (
        SELECT
            username
        FROM ratings
        GROUP BY username
        HAVING COUNT(DISTINCT book_id) > 50
    ) AS u
    LEFT JOIN reviews AS rv ON rv.username = u.username
    GROUP BY u.username
) AS por_usuario;
'''

executar_consulta(query)

,usuarios_com_mais_de_50_classificacoes,media_de_avaliacoes
0,6,24.33


**Result:** six users rated more than 50 books. Among them, the average number of written reviews is **24.33**.

**Conclusion:** the service's most active users are few: only six exceed 50 rated books. These users write, on average, about 24 reviews each. Since all of them rated more than 50 books, this indicates, approximately, that they write a text for less than half of the books they score. The query does not calculate this ratio user by user, so this reading should be taken as an approximation.

Individually, these users are the ones who write the most. As a group, however, the six account for about 146 of the 2,793 reviews in the database, about 5%. This suggests that the bigger opportunity for the new product lies in getting the many less active users to write reviews, not only in encouraging those who already write.

## 4. Conclusions

This study analyzed the database of a competing reading service, with 1,000 books, 636 authors, 340 publishers, 6,456 ratings and 2,793 reviews. All results were obtained with SQL queries.

### Results

| Question | Result |
|---|---|
| Books released after January 1, 2000 | 819 books, 81.9% of the catalog |
| Reviews and average rating per book | 0 to 7 reviews per book, averaging 2.79. The average rating is 3.90 |
| Publisher with the most books over 50 pages | Penguin Books, with 42 books |
| Author with the highest average rating | J.K. Rowling/Mary GrandPré, averaging 4.28 across four books |
| Average reviews among users with more than 50 rated books | 24.33 reviews, among six users |

### What the results suggest for the new product

The results describe a competitor's catalog and user behavior. The recommendations below are working hypotheses, and confirming them would require further analysis, with demand, usage or sales data.

- **Catalog:** most of the competitor's catalog consists of editions published after 2000. This suggests that recent titles should carry significant weight in a competitive catalog.
- **Publishing partnerships:** Penguin Books leads in number of titles, and Penguin Classics is in the top five. This suggests that the publishers at the top of the list are natural candidates for catalog partnerships.
- **Authors:** among books with at least 50 ratings, the best-rated authors write mass-market fiction. The database has no genre information, so a reading by literary genre would require additional data.
- **Popularity and approval:** the book with the most reviews, *Twilight*, has an average rating below the overall average. Review volume and rating are different measures and should be tracked separately.
- **Engagement:** users give scores much more often than they write reviews. There are 6,456 ratings for 2,793 reviews, and, approximately, even the most active users write about fewer than half of the books they rate. Since the six most active users account for only about 5% of reviews, the suggested opportunity lies in encouraging less active users to write.

### Limitations

- The database records similar names as different publishers, which may understate concentration by publishing group.
- In some records, the author field holds more than one name, so the rating is attributed to the group rather than to each person.
- Only six users rated more than 50 books. The average calculated for this group rests on few people.